In [3]:
import pandas as pd
import numpy as np

def add_features(df):
    epsilon = 1e-6
    # 1. 预期威胁时间
    df['projected_time_to_threat'] = df['dist_min_ci_0_5h'] / (df['centroid_speed_m_per_h'] + epsilon)
    # 2. 动能动量指数
    df['kinetic_momentum'] = df['area_growth_rate_ha_per_h'] * df['dist_accel_m_per_h2']
    # 3. 定向威胁速度
    df['directed_speed_vector'] = df['alignment_cos'] * df['centroid_speed_m_per_h']
    # 4. 初始活力指数
    df['initial_vigor_index'] = df['log1p_area_first'] * df['log1p_growth']

    # 填充可能产生的极值
    cols_new = ['projected_time_to_threat', 'kinetic_momentum', 'directed_speed_vector', 'initial_vigor_index']
    for col in cols_new:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(df[col].mean())
    return df

# 严格使用 B 洗好的两个 Clean 版本文件！
train_clean = pd.read_csv('train_clean.csv')
test_clean = pd.read_csv('test_clean.csv')

# 加工
train_aug = add_features(train_clean)
test_aug = add_features(test_clean)

# 确保训练集和测试集的特征列完全一致（除了训练集多出来的 event 和 time 标签）
# 这一步非常关键，模型预测时如果列对不上会报错
features_to_keep = [col for col in train_aug.columns if col not in ['event', 'time_to_hit_hours']]
# 测试集只保留和训练集特征一致的列
test_aug = test_aug[features_to_keep]

# 保存
train_aug.to_csv('train_augmented_D.csv', index=False)
test_aug.to_csv('test_augmented_D.csv', index=False)
print("D 的双重弹药已准备就绪！")

D 的双重弹药已准备就绪！
